In [ ]:
import polars as pl
from polars import col as c
from ucr_tools.core import data_analysis as da 
from definitions import bases
from codigo import transformations as trans

In [ ]:
file_cr = bases.joinpath('cr/ENAHO_2023.sav')
file_chile = bases.joinpath('chile/casen_cleaned_2022.sav')

In [ ]:
df, var_labels, col_labels = da.read_file(file_cr)
df:pl.DataFrame
da.var_labels=var_labels

#### Participacion

In [ ]:
da.apply_labels(
    da.tab(df.filter(c.condact.is_not_null(), c.a4.eq(1), c.lugnac.is_in([0,1])), col='condact', factor='factor'),
    cols='condact'
)

#### Formalidad

In [ ]:
formal_dict = {0:'Informal', 1:'Formal'}
asegurados_list = [1, 8, 9, 12, 14]
df = df.with_columns(pl.when(c.a11.is_in(asegurados_list)).then(1).when(c.a11.is_null() | c.a11.eq(99)).then(None).otherwise(0).alias('formal_flag'))

ocupations = (
    df
    .filter(c.condact.eq(1), c.a4.eq(2))
    .group_by('ramaemppri formal_flag'.split()).agg(c.factor.sum())
    .with_columns(c.factor.truediv(c.factor.sum().over('ramaemppri')).round(2))
    .with_columns(c.formal_flag.replace_strict(formal_dict))
    .pivot(on='formal_flag', index='ramaemppri')
    .sort('ramaemppri')
)
da.apply_labels(ocupations, 'ramaemppri')

#### Perfil mujeres Nicaraguenses y Costarricenses

In [ ]:
df, var_labels, col_labels = da.read_file(bases.joinpath('cr/limpio/cr_region_grouped_processed.sav'))
da.var_labels=var_labels
df:pl.DataFrame
df = (
    df
    .with_columns(
        pl.when(c.ramaemppri.is_in([9,7,20,3])).then('ramaemppri')
        .when(c.ramaemppri.is_null() | c.ramaemppri.eq(99)).then(None)
        .otherwise(-1)
        .alias('rama_2')
    )
)
var_labels['ramaemppri'].update({-1:'Otro'})
var_labels['rama_2'] = var_labels['ramaemppri']

In [114]:
filters = (
    c.a4.eq(2),
    c.nacionalidad.is_in([0,1]),
    c.logro.eq(4), #Cambiar perfil con este filtro
    c.rama_2.is_not_null(),
    c.anno.eq(2023)
)

da.apply_labels(
    da.tab(df.filter(filters), col='rama_2', groups='nacionalidad', factor='factor'),
    cols='nacionalidad rama_2'.split()
).pivot('nacionalidad', index='rama_2').write_clipboard()